# pynq_puf3 实板测试

本 Notebook 是可独立执行的单文件板测入口，不动态引用本目录下的 Python 文件。
它会执行 Overlay 身份检查、SW/HW/ISE KAT、mcycle benchmark 和 PUF enroll/reconstruct/HWKEY 流程，
并把完整证据保存到 `data/pynq_puf3/notebook_<时间>/`。

请在 PYNQ 板上确认 candidate 目录和仓库路径可访问，然后按顺序执行全部单元。

时钟说明：原版 PYNQ 2.7 的 `pynq.Clocks.fclk0_mhz` 固定按 50 MHz PS 参考时钟换算，本板晶振为 33.333333 MHz，所以修正后的 overlay 会显示 150 MHz，这不是实际频率。本 Notebook 改为：下载前核对 IO PLL、下载后按 hwh 晶振由 SLCR 推算 FCLK0，并用 RV32 mcycle 对 PS 时间实测 FCLK0；后两者都必须为 100 MHz。

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import os
import platform
import sys

REPO = Path(os.environ.get("ASCON_REPO", "/home/xilinx/jupyter_notebooks/amd_fpga_2026")).resolve()
CANDIDATE = REPO / "board/vivado_zynq7000_xc7z020_pynq_puf3/candidate"
FULL_KAT = False
WARMUP = 5
REPETITIONS = 30
KAT_COUNT = 529
RECONSTRUCTS = 5
CLOCK_INTERVAL_S = 3.0   # mcycle 时钟实测的两次采样间隔

assert (REPO / "src/pynq/ascon_host_api.py").is_file(), f"仓库路径不正确：{REPO}; 请设置 ASCON_REPO"
assert (CANDIDATE / "pynq_puf3.bit").is_file(), f"找不到 candidate bit：{CANDIDATE}"
assert (CANDIDATE / "pynq_puf3.hwh").is_file(), f"找不到 candidate hwh：{CANDIDATE}"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = REPO / "data/pynq_puf3" / f"notebook_{run_id}"
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src/pynq"))
sys.path.insert(0, str(REPO))

print("repository:", REPO)
print("candidate:", CANDIDATE)
print("output:", OUT)
print("python:", platform.python_version())


repository: /home/xilinx/jupyter_notebooks/amd_fpga_2026
candidate: /home/xilinx/jupyter_notebooks/amd_fpga_2026/board/vivado_zynq7000_xc7z020_pynq_puf3/candidate
output: /home/xilinx/jupyter_notebooks/amd_fpga_2026/data/pynq_puf3/notebook_20261030_123118
python: 3.8.2


In [ ]:
"""PYNQ PUF3 overlay —— pynq_puf3 overlay 的 PS 侧驱动：三模式 Y01 服务 + PUF v1 帧。

在 pynq_transport.PynqUartTransport 之上只增加两件事（发布版 pynq_transport.py 不改）：
  * 控制窗口身份 "PQF3"（0x33465150）；发布版 "PYNQ" 与 PUF 自检版 "PFU1" 都会被拒绝；
  * Y01P 请求 / Y01K 响应（固件 src/firmware/pynq_svc_puf/puf_svc.c）。

Y01P 16B: b'Y01P' | u32 version | u32 cmd | u32 seq
Y01K 112B: magic | u16 version | u16 flags | u32 cmd | u32 seq | u32 result | u32 STATUS
           | u32 DNA_LO | u32 DNA_HI | u64 op mcycle | REF16 | MASK16 | RAW16 | DITHER16 | u64 now mcycle
（now mcycle = 固件发帧时刻的 mcycle；两帧的 Δmcycle/Δwall 即 CPU 实际时钟，见 measure_cpu_clock_mhz）

证据边界：REF/MASK/RAW/DITHER/DNA 是 PUF v1 设计上可读的诊断量，不能据此宣称保密；
key 本身不可读。HWKEY 打开后 HW 模式用 PUF 派生密钥，结果无法与官方 KAT 比对，
只能用"调用方 key 被拒绝 + 同 key 往返 + 重建后一致"来证明。
"""
from __future__ import annotations

import struct

if __package__:
    from src.pynq.ascon_host_api import BackendProtocolError, BackendTimeoutError
    from src.pynq.mailbox_protocol import VERSION
    from src.pynq.pynq_transport import (BridgeRegs, CTL_ID, CTL_VERSION, FrameDecoder, MB_ID, MB_ID_VALUE, NotConfiguredError, PynqUartTransport, RSP_HEADER_BYTES, SWC_FRAME_BYTES, _need_finite, parse_service_frame)
else:
    from ascon_host_api import BackendProtocolError, BackendTimeoutError
    from mailbox_protocol import VERSION
    from pynq_transport import (BridgeRegs, CTL_ID, CTL_VERSION, FrameDecoder, MB_ID, MB_ID_VALUE, NotConfiguredError, PynqUartTransport, RSP_HEADER_BYTES, SWC_FRAME_BYTES, _need_finite, parse_service_frame)

CTL_ID_PQF3 = 0x33465150
FIRMWARE_ID = 'pynq-svc-puf3'
PUF_RSP_BYTES = 112

CMD = {'status': 0, 'enroll': 1, 'reconstruct': 2, 'hwkey_on': 3, 'hwkey_off': 4}
RESULT = {0: 'OK', 1: 'TIMEOUT', 2: 'NOT_ACCEPTED', 3: 'BAD_CMD', 4: 'BAD_VERSION', 5: 'NO_PUF_KEY', 6: 'BUSY_STUCK'}
STATUS_BITS = ('busy', 'key_ready', 'enrolled', 'mismatch', 'fallback', 'done', 'key_hw_sel', 'hwkey_on')


class Puf3BridgeRegs(BridgeRegs):
    def check_identity(self):
        ident = self.ctl.read(CTL_ID)
        if ident != CTL_ID_PQF3:
            raise NotConfiguredError('NOT_CONFIGURED: control identity 0x%08X is not PQF3' % ident)
        ident = self.mb.read(MB_ID)
        if ident != MB_ID_VALUE:
            raise NotConfiguredError('NOT_CONFIGURED: mailbox window identity mismatch (0x%08X)' % ident)
        ver = self.ctl.read(CTL_VERSION)
        if ver != 1:
            raise NotConfiguredError('NOT_CONFIGURED: control block version %r' % ver)
        return True


class Puf3FrameDecoder(FrameDecoder):
    """Y01S / Y01R / Y01K；其余字节当文本日志。"""

    MAGICS = ((b'Y01S', 'S'), (b'Y01R', 'R'), (b'Y01K', 'K'))

    def _find_magic(self):
        best, kind = -1, None
        for magic, tag in self.MAGICS:
            idx = self.buf.find(magic)
            if idx >= 0 and (best < 0 or idx < best):
                best, kind = idx, tag
        return best, kind

    def pop_frame(self):
        """返回 ('S'|'R'|'K', bytes) 或 None；与基类相同的分帧规则，多一种定长 K 帧。"""
        while True:
            idx, kind = self._find_magic()
            if idx < 0:
                if len(self.buf) > 3:
                    self.log.extend(self.buf[:-3])
                    del self.buf[:-3]
                return None
            if idx:
                self.log.extend(self.buf[:idx])
                del self.buf[:idx]
            if kind in ('S', 'K'):
                size = SWC_FRAME_BYTES if kind == 'S' else PUF_RSP_BYTES
                if len(self.buf) < size:
                    return None
                frame = bytes(self.buf[:size])
                del self.buf[:size]
                return kind, frame
            if len(self.buf) < RSP_HEADER_BYTES:
                return None
            size = int.from_bytes(self.buf[36:40], 'little')
            if size > 4096:
                self.log.extend(self.buf[:1])
                del self.buf[:1]
                continue
            total = RSP_HEADER_BYTES + size
            if len(self.buf) < total:
                return None
            frame = bytes(self.buf[:total])
            del self.buf[:total]
            return 'R', frame


def puf_request(cmd, seq):
    # Any u32 is encodable on purpose: the firmware must reject unknown commands (BAD_CMD).
    if isinstance(cmd, bool) or not isinstance(cmd, int) or not 0 <= cmd < 2 ** 32:
        raise ValueError('invalid PUF command')
    return b'Y01P' + struct.pack('<III', VERSION, cmd, seq)


def parse_puf_frame(frame):
    if len(frame) != PUF_RSP_BYTES or frame[:4] != b'Y01K':
        raise BackendProtocolError('malformed PUF frame')
    version, flags, cmd, seq, result, status, dna_lo, dna_hi, cycles = struct.unpack_from('<HHIIIIIIQ', frame, 4)
    if version != VERSION:
        raise BackendProtocolError('PUF frame version mismatch')
    now = struct.unpack_from('<Q', frame, 104)[0]
    if now == 0:
        raise BackendProtocolError('PUF frame carries no mcycle timestamp')
    bits = {name: bool(status >> i & 1) for i, name in enumerate(STATUS_BITS)}
    return {
        'cmd': cmd, 'seq': seq, 'result': RESULT.get(result, 'UNKNOWN_%d' % result),
        'result_code': result, 'simulated': bool(flags & 0x2), 'status_raw': status,
        'status': bits, 'quality': (status >> 8) & 0xFF,
        'dna': '%07x%08x' % (dna_hi & 0x1FFFFFF, dna_lo),
        'op_mcycle': cycles, 'now_mcycle': now,
        'ref': frame[40:56].hex(), 'mask': frame[56:72].hex(),
        'raw': frame[72:88].hex(), 'dither': frame[88:104].hex(),
    }


class PynqPuf3Transport(PynqUartTransport):
    """PynqUartTransport + Y01K 分帧 + puf_command()。"""

    def __init__(self, regs, *, expected_firmware_id=FIRMWARE_ID, **kwargs):
        if not isinstance(regs, Puf3BridgeRegs):
            raise NotConfiguredError('NOT_CONFIGURED: Puf3BridgeRegs instance required')
        self._puf_frame = None
        self._puf_seq = 0
        super().__init__(regs, expected_firmware_id=expected_firmware_id, **kwargs)

    # The base class assigns `self.decoder = FrameDecoder()` in __init__ and on every handshake;
    # route every assignment to a decoder that also understands Y01K.
    @property
    def decoder(self):
        return self._decoder

    @decoder.setter
    def decoder(self, value):
        self._decoder = value if isinstance(value, Puf3FrameDecoder) else Puf3FrameDecoder()

    def _pump(self, deadline, want='R'):
        while True:
            self._check(deadline)
            chunk = self.regs.drain()
            if chunk:
                self.decoder.feed(chunk)
            while True:
                item = self.decoder.pop_frame()
                if item is None:
                    break
                kind, frame = item
                if kind == 'S':
                    self.last_service = parse_service_frame(frame)
                    if want == 'S':
                        return frame
                    continue
                if kind == 'K':
                    if want == 'K':
                        return frame
                    raise BackendProtocolError('unexpected PUF frame')
                if want == 'R':
                    return frame
                raise BackendProtocolError('unexpected response frame')
            if self.regs.rx_overflow():
                raise BackendProtocolError('mailbox RX overflow: PS did not drain in time')
            if self.clock() >= deadline:
                raise BackendTimeoutError('pynq transport deadline expired')
            self.sleep(min(self.poll_interval, max(0.0, deadline - self.clock())))

    def puf_command(self, name, deadline):
        """执行一条 PUF 命令并返回解析后的 Y01K；不能与未完成的 AEAD 请求交错。"""
        _need_finite(deadline)
        if self.capabilities is None:
            raise NotConfiguredError('NOT_CONFIGURED: hello() first')
        if self._pending is not None or self._response is not None:
            raise BackendProtocolError('mailbox slot occupied by an AEAD request')
        cmd = CMD[name] if isinstance(name, str) else int(name)
        self._puf_seq = (self._puf_seq + 1) & 0xFFFFFFFF or 1
        seq = self._puf_seq
        self._push(puf_request(cmd, seq), deadline)
        frame = parse_puf_frame(self._pump(deadline, want='K'))
        frame['host_monotonic'] = self.clock()
        if (frame['cmd'], frame['seq']) != (cmd, seq):
            raise BackendProtocolError('stale or mismatched PUF response')
        if frame['simulated'] != bool(self.capabilities.simulated):
            raise BackendProtocolError('PUF frame evidence kind mismatch')
        return frame


def measure_cpu_clock_mhz(transport, interval_s=3.0, *, sleep=None, timeout_s=10.0):
    """用两次 STATUS 帧的 Δmcycle / Δ主机单调时间测出 RV32 实际时钟（= FCLK0）。

    与 SLCR 寄存器推算无关的独立测量；误差主要来自帧到达时刻抖动（约 1 ms 量级），
    interval_s=3 时约 0.03 %。返回 (mhz, 两帧原始值)。
    """
    if not interval_s > 0:
        raise ValueError('interval_s must be positive')
    sleep = sleep or transport.sleep
    a = transport.puf_command('status', transport.clock() + timeout_s)
    sleep(interval_s)
    b = transport.puf_command('status', transport.clock() + timeout_s)
    dt = b['host_monotonic'] - a['host_monotonic']
    dc = b['now_mcycle'] - a['now_mcycle']
    if dt <= 0 or dc <= 0:
        raise BackendProtocolError('clock measurement went backwards')
    return dc / dt / 1e6, {'mcycle': [a['now_mcycle'], b['now_mcycle']],
                           'host_s': [a['host_monotonic'], b['host_monotonic']]}


# ---- Zynq-7000 SLCR（UG585 附录 B）：只读，用于按真实 PS 参考时钟推算 FCLK0 ----
SLCR_BASE = 0xF8000000
SLCR_ARM_PLL_CTRL = 0x100
SLCR_DDR_PLL_CTRL = 0x104
SLCR_IO_PLL_CTRL = 0x108
SLCR_FPGA0_CLK_CTRL = 0x170


def pll_fbdiv(ctrl):
    return (ctrl >> 12) & 0x7F


def fclk0_from_registers(crystal_mhz, regs):
    """regs: {'io','arm','ddr','fpga0'} 原始寄存器值；返回 (mhz, 明细)。

    与 pynq.Clocks 不同，这里用 hwh 中 PCW_CRYSTAL_PERIPHERAL_FREQMHZ 作为参考时钟，
    不使用 PYNQ 2.7 写死的 50 MHz 假设。PLL 处于 bypass 时直接用参考时钟。
    """
    fpga0 = regs['fpga0']
    srcsel = (fpga0 >> 4) & 0x3
    div0 = (fpga0 >> 8) & 0x3F
    div1 = (fpga0 >> 20) & 0x3F
    name, ctrl = {0: ('IO', regs['io']), 1: ('IO', regs['io']), 2: ('ARM', regs['arm']),
                  3: ('DDR', regs['ddr'])}[srcsel]
    bypass = bool(ctrl & 0x10)
    pll = crystal_mhz if bypass else crystal_mhz * pll_fbdiv(ctrl)
    if div0 == 0 or div1 == 0:
        raise ValueError('FPGA0 divisor is zero')
    return pll / div0 / div1, {'source_pll': name, 'pll_fbdiv': pll_fbdiv(ctrl), 'pll_bypass': bypass,
                               'pll_mhz': pll, 'divisor0': div0, 'divisor1': div1,
                               'crystal_mhz': crystal_mhz}


def read_slcr(mmio):
    return {'io': mmio.read(SLCR_IO_PLL_CTRL), 'arm': mmio.read(SLCR_ARM_PLL_CTRL),
            'ddr': mmio.read(SLCR_DDR_PLL_CTRL), 'fpga0': mmio.read(SLCR_FPGA0_CLK_CTRL)}


def hwh_ps_clock(hwh_text):
    """从 hwh 取 PS 参考时钟、IO PLL FBDIV 与 FCLK0 分频（build 时的假设）。"""
    import re

    def p(name):
        m = re.search(r'NAME="%s" VALUE="([^"]*)"' % name, hwh_text)
        if not m:
            raise ValueError('hwh lacks ' + name)
        return m.group(1)
    return {'crystal_mhz': float(p('PCW_CRYSTAL_PERIPHERAL_FREQMHZ')),
            'iopll_fbdiv': int(p('PCW_IOPLL_CTRL_FBDIV')),
            'fclk0_src': p('PCW_FCLK0_PERIPHERAL_CLKSRC'),
            'divisor0': int(p('PCW_FCLK0_PERIPHERAL_DIVISOR0')),
            'divisor1': int(p('PCW_FCLK0_PERIPHERAL_DIVISOR1')),
            'act_fclk0_mhz': float(p('PCW_ACT_FPGA0_PERIPHERAL_FREQMHZ'))}


def from_overlay(overlay, control_name='pynq_ctl_0', mailbox_name='pynq_mbox_0', **kwargs):
    if overlay is None or not callable(getattr(overlay, 'is_loaded', None)) or not overlay.is_loaded():
        raise NotConfiguredError('NOT_CONFIGURED: caller must provide a loaded overlay')
    ip_dict = getattr(overlay, 'ip_dict', {})
    for name in (control_name, mailbox_name):
        if name not in ip_dict:
            raise NotConfiguredError('NOT_CONFIGURED: %r missing from overlay ip_dict' % name)
    return PynqPuf3Transport(Puf3BridgeRegs(getattr(overlay, control_name).mmio, getattr(overlay, mailbox_name).mmio), **kwargs)


__all__ = ['CTL_ID_PQF3', 'FIRMWARE_ID', 'CMD', 'RESULT', 'Puf3BridgeRegs', 'Puf3FrameDecoder', 'PynqPuf3Transport', 'parse_puf_frame', 'puf_request', 'from_overlay', 'measure_cpu_clock_mhz', 'fclk0_from_registers', 'read_slcr', 'hwh_ps_clock', 'pll_fbdiv', 'SLCR_BASE', 'RSP_HEADER_BYTES', 'SWC_FRAME_BYTES']


In [ ]:
import datetime
import hashlib
import json
import platform
import time
from pathlib import Path

from ascon_host_api import AsconHost, AuthenticationError
from board_benchmark import _run_once, select_kat_case, summarize
from golden_checker import parse_kat
from mailbox_backend import MailboxBackend
from mailbox_protocol import Mode

MODES = (('sw', Mode.SW), ('hw', Mode.HW), ('ise', Mode.ISE))


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def set_mode(backend, mode):
    backend.mode = mode
    backend.name = 'mailbox-' + mode.name.lower()


def section_kat(backend, full):
    cases = [c for c in parse_kat() if len(c['PT']) <= 16 and len(c['AD']) <= 16]
    if not full:
        cases = [cases[i] for i in (0, 1, 16, 17, 144, 288)]
    out = {}
    host = AsconHost(backend)
    for name, mode in MODES:
        set_mode(backend, mode)
        passed = rejected = 0
        for c in cases:
            k, n, ad, pt, ct = (c[x] for x in ('Key', 'Nonce', 'AD', 'PT', 'CT'))
            r = host.encrypt(k, n, ad, pt, timeout_s=10)
            assert not r.simulated and r.ciphertext + r.tag == ct, '%s KAT %s encrypt mismatch' % (name, c['count'])
            assert host.decrypt(k, n, ad, ct[:-16], ct[-16:], timeout_s=10).plaintext == pt
            try:
                host.decrypt(k, n, ad, ct[:-16], ct[-16:-1] + bytes([ct[-1] ^ 1]), timeout_s=10)
            except AuthenticationError:
                rejected += 1
            else:
                raise AssertionError('%s accepted a bad tag' % name)
            passed += 1
        out[name] = {'kat_pass': passed, 'tag_reject': rejected, 'cases': [c['count'] for c in cases]}
        print('kat', name, out[name]['kat_pass'], flush=True)
    return out


def section_benchmark(backend, warmup, reps, kat_count):
    case = select_kat_case(kat_count)
    host = AsconHost(backend)
    out = {'kat': kat_count, 'warmup': warmup, 'repetitions': reps,
           'measurement_scope': 'firmware AEAD mcycle; excludes PS/UART', 'modes': {}}
    for name, mode in MODES:
        set_mode(backend, mode)
        for _ in range(warmup):
            _run_once(host, case)
        samples = [_run_once(host, case) for _ in range(reps)]
        out['modes'][name] = {}
        for op in ('encrypt', 'decrypt'):
            raw = [s[op + '_cycles'] for s in samples]
            wall = [s[op + '_wall_ns'] for s in samples]
            out['modes'][name][op] = {'summary': summarize(raw), 'raw_cycles': raw,
                                      'wall_ns_summary': summarize(wall)}
        print('bench', name, {op: out['modes'][name][op]['summary']['median'] for op in ('encrypt', 'decrypt')},
              flush=True)
    sw = out['modes']['sw']
    out['speedup_median_vs_sw'] = {
        name: {op: round(sw[op]['summary']['median'] / out['modes'][name][op]['summary']['median'], 3)
               for op in ('encrypt', 'decrypt')} for name in ('hw', 'ise')}
    print('speedup', out['speedup_median_vs_sw'], flush=True)
    return out


def section_puf(transport, backend, reconstructs):
    log = []

    def cmd(name, expect='OK'):
        frame = transport.puf_command(name, time.monotonic() + 10)
        log.append(frame)
        print('puf', name, frame['result'], hex(frame['status_raw']), 'quality', frame['quality'], flush=True)
        assert frame['result'] == expect, '%s returned %s' % (name, frame['result'])
        return frame

    case = select_kat_case()
    k, n, ad, pt, ct = (case[x] for x in ('Key', 'Nonce', 'AD', 'PT', 'CT'))
    host = AsconHost(backend)
    set_mode(backend, Mode.HW)
    boot = cmd('status')
    if boot['status']['fallback']:
        cmd('hwkey_on', expect='NO_PUF_KEY')       # DNA-only fallback key must not be accepted
    enroll = cmd('enroll')
    assert enroll['status']['enrolled'] and enroll['quality'] > 0, 'enroll produced no usable bits'
    recon = []
    for _ in range(reconstructs):
        f = cmd('reconstruct')
        s = f['status']
        assert s['key_ready'] and not s['mismatch'] and not s['fallback'], 'reconstruct mismatch/fallback'
        assert (f['ref'], f['mask'], f['dna']) == (enroll['ref'], enroll['mask'], enroll['dna'])
        recon.append(f)
    on = cmd('hwkey_on')
    assert on['status']['key_hw_sel'] and on['status']['hwkey_on']
    r1 = host.encrypt(k, n, ad, pt, timeout_s=10)
    assert r1.ciphertext + r1.tag != ct, 'HW still used the caller key with HWKEY on'
    assert host.decrypt(k, n, ad, r1.ciphertext, r1.tag, timeout_s=10).plaintext == pt, 'PUF-key round trip'
    try:
        host.decrypt(k, n, ad, ct[:-16], ct[-16:], timeout_s=10)
    except AuthenticationError:
        caller_key_rejected = True
    else:
        raise AssertionError('official KAT ciphertext accepted under the PUF key')
    cmd('reconstruct')                               # firmware switches HWKEY off while regenerating
    cmd('hwkey_on')
    r2 = host.encrypt(k, n, ad, pt, timeout_s=10)
    assert (r2.ciphertext, r2.tag) == (r1.ciphertext, r1.tag), 'PUF key changed after reconstruct'
    off = cmd('hwkey_off')
    assert not off['status']['key_hw_sel']
    r3 = host.encrypt(k, n, ad, pt, timeout_s=10)
    assert r3.ciphertext + r3.tag == ct, 'HW KAT not restored after HWKEY off'
    return {'frames': log, 'dna': enroll['dna'], 'enroll_quality': enroll['quality'],
            'reconstructs_stable': len(recon), 'caller_key_rejected_under_puf_key': caller_key_rejected,
            'puf_key_ciphertext_tag': (r1.ciphertext + r1.tag).hex(),
            'same_ciphertext_after_reconstruct': True, 'kat_restored_after_hwkey_off': True,
            'puf_key_hw_encrypt_cycles': r1.cycles,
            'limits': 'in-session only (helper volatile); REF/MASK/RAW/DNA readable; '
                      'entropy/BER/temperature/cross-power-cycle not measured'}



In [ ]:
FCLK_TARGET_MHZ = 100.0


def run_board_test(candidate, out, *, full_kat=False, warmup=5, repetitions=30, kat_count=529, reconstructs=5,
                   clock_interval_s=3.0):
    if warmup < 0 or repetitions < 2 or reconstructs < 1 or clock_interval_s < 1:
        raise ValueError("warmup >= 0, repetitions >= 2, reconstructs >= 1, clock_interval_s >= 1 required")
    out.mkdir(parents=True, exist_ok=False)
    bit = candidate / "pynq_puf3.bit"
    hwh = candidate / "pynq_puf3.hwh"
    manifest_path = candidate / "manifest.json"
    result = {
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "kind": "physical_board_pynq_puf3",
        "host": platform.platform(),
        "python": sys.version,
        "command": ["notebook", "run_board_test"],
        "sections": {},
        "status": "FAIL",
        "exit_code": 1,
        "timing_signoff_ro_domain": False,
        "cross_power_cycle": "not_measured",
    }
    transport = None
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        expected = manifest["artifacts"]
        result.update(
            bit_sha256=digest(bit),
            hwh_sha256=digest(hwh),
            manifest_sha256=digest(manifest_path),
        )
        if (result["bit_sha256"] != expected["pynq_puf3.bit"]["sha256"]
                or result["hwh_sha256"] != expected["pynq_puf3.hwh"]["sha256"]):
            raise RuntimeError("bit/hwh do not match candidate manifest")
        import pynq
        result["pynq_version"] = pynq.__version__
        # FCLK0 = board IO PLL / hwh divisors. PYNQ only writes the divisors; the IO PLL comes from
        # the boot image, so the hwh's IO PLL assumption must match the running PLL.
        ps = hwh_ps_clock(hwh.read_text(encoding="utf-8", errors="replace"))
        slcr = pynq.MMIO(SLCR_BASE, 0x200)
        before = read_slcr(slcr)
        result["clock"] = {"hwh": ps, "slcr_before_download": {k: "0x%08X" % v for k, v in before.items()}}
        if pll_fbdiv(before["io"]) != ps["iopll_fbdiv"] or before["io"] & 0x10:
            raise RuntimeError("board IO PLL FBDIV %d (bypass=%d) != hwh %d: FCLK0 would not be %.0f MHz; "
                               "refusing to download" % (pll_fbdiv(before["io"]), bool(before["io"] & 0x10),
                                                         ps["iopll_fbdiv"], FCLK_TARGET_MHZ))
        overlay = pynq.Overlay(str(bit), download=True)
        result["ip_dict"] = {
            key: {"phys_addr": value.get("phys_addr"), "addr_range": value.get("addr_range")}
            for key, value in overlay.ip_dict.items()
        }
        for name in ("pynq_ctl_0", "pynq_mbox_0"):
            if name not in overlay.ip_dict:
                raise RuntimeError("HWH instance missing: " + name)
            for field in ("phys_addr", "addr_range"):
                if field not in overlay.ip_dict[name]:
                    raise RuntimeError("HWH instance %s missing metadata: %s" % (name, field))
        ctl = pynq.MMIO(overlay.ip_dict["pynq_ctl_0"]["phys_addr"], overlay.ip_dict["pynq_ctl_0"]["addr_range"])
        mailbox = pynq.MMIO(overlay.ip_dict["pynq_mbox_0"]["phys_addr"], overlay.ip_dict["pynq_mbox_0"]["addr_range"])
        after = read_slcr(slcr)
        fclk_reg, detail = fclk0_from_registers(ps["crystal_mhz"], after)
        result["clock"].update(
            slcr_after_download={k: "0x%08X" % v for k, v in after.items()},
            fclk0_mhz_register_derived=round(fclk_reg, 6), register_detail=detail,
            # stock PYNQ 2.7 _ClocksZynq assumes a 50 MHz PS reference; the board crystal is in the hwh
            pynq_clocks_fclk0_mhz=float(pynq.Clocks.fclk0_mhz),
            pynq_reference_clock_mhz_assumed=getattr(getattr(pynq.Clocks, "_instance", None), "_ref_clk_mhz", None))
        if abs(fclk_reg - FCLK_TARGET_MHZ) > 0.5:
            raise RuntimeError("register-derived FCLK0 %.3f MHz != %.0f MHz" % (fclk_reg, FCLK_TARGET_MHZ))
        regs = Puf3BridgeRegs(ctl, mailbox)
        regs.check_identity()
        result["control_id"] = "0x%08X" % CTL_ID_PQF3
        transport = PynqPuf3Transport(regs, expected_firmware_id=FIRMWARE_ID)
        backend = MailboxBackend(transport, mode="sw", timeout_s=30.0)
        result["service"] = transport.last_service
        # Independent measurement: RV32 mcycle runs on FCLK0; compare with PS monotonic time.
        mhz, raw = measure_cpu_clock_mhz(transport, clock_interval_s)
        result["clock"].update(fclk0_mhz_measured_mcycle=round(mhz, 4), measurement=raw,
                               measurement_interval_s=clock_interval_s)
        print("FCLK0 register-derived %.3f MHz, measured %.3f MHz (PYNQ Clocks shows %s with %s MHz ref)"
              % (fclk_reg, mhz, result["clock"]["pynq_clocks_fclk0_mhz"],
                 result["clock"]["pynq_reference_clock_mhz_assumed"]), flush=True)
        if abs(mhz - FCLK_TARGET_MHZ) > 0.01 * FCLK_TARGET_MHZ:
            raise RuntimeError("measured CPU clock %.3f MHz differs from %.0f MHz by more than 1%%"
                               % (mhz, FCLK_TARGET_MHZ))
        result["sections"]["kat"] = section_kat(backend, full_kat)
        result["sections"]["benchmark"] = section_benchmark(backend, warmup, repetitions, kat_count)
        result["sections"]["puf"] = section_puf(transport, backend, reconstructs)
        result.update(status="PASS", exit_code=0)
    except Exception as error:
        result["error"] = repr(error)
    finally:
        if transport is not None:
            snapshot = transport.status_snapshot()
            (out / "mailbox_log.txt").write_text(snapshot.pop("log"), encoding="utf-8")
            result["final_snapshot"] = snapshot
        (out / "result.json").write_text(json.dumps(result, indent=2, default=str) + "\n", encoding="utf-8")
    print(result["status"], out, result.get("error", ""))
    return result


## 执行完整上板测试

默认使用 6 个代表性 KAT。将 `FULL_KAT = True` 后重新执行配置单元，可改为运行全部 289 个容量内 KAT。
测试失败时也会先保存 `result.json` 和 `mailbox_log.txt`，便于定位问题。

In [ ]:
result = run_board_test(
    CANDIDATE, OUT, full_kat=FULL_KAT, warmup=WARMUP,
    repetitions=REPETITIONS, kat_count=KAT_COUNT, reconstructs=RECONSTRUCTS,
    clock_interval_s=CLOCK_INTERVAL_S,
)
print(json.dumps({
    "status": result.get("status"),
    "exit_code": result.get("exit_code"),
    "output": str(OUT),
    "error": result.get("error"),
}, indent=2, ensure_ascii=False))
if result.get("exit_code") != 0:
    raise RuntimeError(f"pynq_puf3 board test failed; evidence: {OUT}")


FAIL /home/xilinx/jupyter_notebooks/amd_fpga_2026/data/pynq_puf3/notebook_20261030_123118 BackendProtocolError('unexpected PUF frame')
{
  "status": "FAIL",
  "exit_code": 1,
  "output": "/home/xilinx/jupyter_notebooks/amd_fpga_2026/data/pynq_puf3/notebook_20261030_123118",
  "error": "BackendProtocolError('unexpected PUF frame')"
}


RuntimeError: pynq_puf3 board test failed; evidence: /home/xilinx/jupyter_notebooks/amd_fpga_2026/data/pynq_puf3/notebook_20261030_123118

In [ ]:
print("status:", result["status"])
print("output directory:", OUT)
print("sections:")
for name, section in result.get("sections", {}).items():
    print(f"- {name}: PASS")
print("\n完整证据文件：")
for path in sorted(OUT.iterdir()):
    print("-", path.name)
print("\n限制：跨断电重建、熵、BER、温度和唯一性仍为 not_measured。")


status: FAIL
output directory: /home/xilinx/jupyter_notebooks/amd_fpga_2026/data/pynq_puf3/notebook_20261030_123118
sections:

完整证据文件：
- mailbox_log.txt
- result.json

限制：跨断电重建、熵、BER、温度和唯一性仍为 not_measured。
